# Part C: Regularized Linear Models

Ridge (L2), Lasso (L1), Hyperparameter Tuning, and Model Comparison.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv('Advanced Regression HousePrice.csv')
df.head()


,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


In [2]:
if 'sale_date' in df.columns:
    df['sale_date'] = pd.to_datetime(df['sale_date'])
    df['sale_year'] = df['sale_date'].dt.year
    df['sale_month'] = df['sale_date'].dt.month
    df.drop('sale_date', axis=1, inplace=True)

target_col = 'house_price_inr'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [3]:
ridge_cv = GridSearchCV(
    Ridge(),
    {'alpha':[0.01,0.1,1,10,100,1000]},
    cv=5,
    scoring='neg_mean_squared_error'
)
ridge_cv.fit(X_train_scaled, y_train)
best_ridge = ridge_cv.best_estimator_
print("Best Ridge:", ridge_cv.best_params_)


Best Ridge: {'alpha': 1}


In [4]:
lasso_cv = GridSearchCV(
    Lasso(max_iter=10000),
    {'alpha':[0.001,0.01,0.1,1,10,100]},
    cv=5,
    scoring='neg_mean_squared_error'
)
lasso_cv.fit(X_train_scaled, y_train)
best_lasso = lasso_cv.best_estimator_
print("Best Lasso:", lasso_cv.best_params_)


Best Lasso: {'alpha': 100}


In [5]:
ridge_pred = best_ridge.predict(X_test_scaled)
lasso_pred = best_lasso.predict(X_test_scaled)

comparison = pd.DataFrame({
    'Model':['Ridge','Lasso'],
    'RMSE':[
        np.sqrt(mean_squared_error(y_test,ridge_pred)),
        np.sqrt(mean_squared_error(y_test,lasso_pred))
    ],
    'R2':[
        r2_score(y_test,ridge_pred),
        r2_score(y_test,lasso_pred)
    ]
})
comparison


,Model,RMSE,R2
0,Ridge,2.540065e+06,0.919887
1,Lasso,2.539854e+06,0.919900


In [6]:
coef_df = pd.DataFrame({
    'Feature':X.columns,
    'Ridge_Coefficient':best_ridge.coef_,
    'Lasso_Coefficient':best_lasso.coef_
})
coef_df


,Feature,Ridge_Coefficient,Lasso_Coefficient
0,property_id,-6.527146e+04,-6.529694e+04
1,area_sqft,6.950206e+06,6.957515e+06
2,bedrooms,2.925828e+05,2.864038e+05
3,bathrooms,2.740168e+05,2.740541e+05
4,location_score,3.681382e+06,3.683439e+06
5,property_age,-6.501791e+05,-6.502685e+05
6,distance_city_km,-2.688562e+04,-2.544449e+04
7,near_school,1.327036e+04,1.309780e+04
8,near_metro,5.281289e+04,5.280225e+04
9,crime_rate_index,-1.398465e+05,-1.396840e+05
